# AXIS paper benchmark explorer

Run a small tiled remap and inspect the MPI and Dask evidence side by side. The default results are synthetic, including the performance and accuracy values. Set `AXIS_BENCHMARK_RESULTS` to an actual result bundle to analyze measured runs. This notebook reads existing results and runs its tiny example with local Dask workers. Live Slurm testing is an explicit shell step at the end.

The environment needs the built AXIS Python package, the paper benchmark dependencies, an IPython kernel, pandas, and Matplotlib. Set `AXIS_PAPER_DIR` if running the notebook outside the HELM checkout.

In [ ]:
import os
import sys
import tempfile
from itertools import chain
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from distributed import Client
from IPython.display import display

# Keep this two-worker demonstration from consuming each worker's CPU allotment.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KOKKOS_NUM_THREADS", "1")

candidates = list(
    chain.from_iterable(
        (root, root / "libs/axis/paper") for root in (Path.cwd(), *Path.cwd().parents)
    )
)
paper_dir = (
    Path(os.environ["AXIS_PAPER_DIR"]).resolve()
    if "AXIS_PAPER_DIR" in os.environ
    else next(
        (path for path in candidates if (path / "scripts/benchmark_summary.py").is_file()), None
    )
)
if paper_dir is None:
    raise FileNotFoundError("Set AXIS_PAPER_DIR to the HELM libs/axis/paper directory.")
sys.path.insert(0, str(paper_dir / "scripts"))
sys.path.insert(0, str(paper_dir / "notebooks/example_results"))

results_input = os.environ.get("AXIS_BENCHMARK_RESULTS")
is_example = not results_input
work_dir = Path(tempfile.mkdtemp(prefix="axis-paper-notebook-"))
results_dir = Path(results_input).resolve() if results_input else work_dir / "example-results"
export_dir = Path(os.environ.get("AXIS_BENCHMARK_EXPORTS", work_dir / "exports")).resolve()
if results_dir == export_dir or results_dir in export_dir.parents:
    raise ValueError("Export to a directory outside the raw evidence bundle.")
export_dir.mkdir(parents=True, exist_ok=True)
evidence_label = "Synthetic example" if is_example else "Selected evidence bundle"
print(f"{evidence_label} · exports: {export_dir}")

In [ ]:
from axis import DistributedRegridder
from benchmark_fixtures import load_dask_fixture, prepare_tiny_fixture
from benchmark_summary import (
    audit_rows,
    load_runs,
    paired_speedups,
    phase_summaries,
    reuse_break_even,
    scaling_series,
)
from build_example import create_example

## Local tiled regridding

Prepare a two-tile spherical identity fixture, load it from shared-format NetCDF shards, then fit and reuse a first-order conservative operator. The destination mask crosses the tile boundary. This cell uses the same fixture interface planned for the live worker test. Numerical validation runs after computing the lazy result.

In [ ]:
manifest = prepare_tiny_fixture(work_dir / "tiny-fixture", ranks=2, masked=True)
fixture = load_dask_fixture(manifest)
with Client(n_workers=2, threads_per_worker=1, processes=False, dashboard_address=None) as client:
    operator = DistributedRegridder(client, method="conservative").fit(
        fixture.source_tiles, fixture.destination_tiles
    )
    result = operator.apply(fixture.field, dim="cell").compute()
    reused = operator.apply(fixture.field * 2, dim="cell").compute()
np.testing.assert_allclose(result.values, fixture.expected, rtol=1e-8, atol=1e-10)
np.testing.assert_allclose(reused.values, 2 * fixture.expected, rtol=1e-8, atol=1e-10)
print(f"Validated {result.size} destination cells and one reuse application.")

## Audit and comparison gates

A timed point must complete and pass its quality checks. Speedups pair allocation-level timings only for matching fixtures, methods, settings, resource layouts, and measurement scopes. Unmatched phases cannot establish a speedup. Failed and unsupported runs stay in the audit and all searches for comparable evidence keep MPI and Python separate. Each valid phase first summarizes trials within launches, then launches within allocations.

In [ ]:
if is_example:
    create_example(results_dir)
runs = load_runs(results_dir)
if not runs:
    raise ValueError(f"No versioned run.json records found in {results_dir}")

audit = pd.DataFrame(audit_rows(runs))
phase = pd.DataFrame(phase_summaries(runs))
pairs = pd.DataFrame(paired_speedups(runs))
scaling = pd.DataFrame(scaling_series(runs))
reuse = pd.DataFrame(reuse_break_even(runs))
failed = audit.loc[audit["comparison"] != "has-paired-results"]
audit_fields = [
    "run_id",
    "pathway",
    "engine",
    "nodes",
    "status",
    "comparison",
    "unmatched_phases",
    "reason",
]
display(failed[audit_fields])
quality_fields = [column for column in audit if column.startswith("metric_")]
if quality_fields:
    display(audit[["run_id", "pathway", "engine", *quality_fields]].head(10))
    errors = [
        column
        for column in (
            "metric_max_abs_error",
            "metric_area_weighted_rmse",
            "metric_rmse",
            "metric_conservation_error",
            "metric_mass_residual",
        )
        if column in quality_fields and audit[column].notna().any()
    ]
    if errors:
        fig, axes = plt.subplots(
            1,
            len(errors),
            squeeze=False,
            figsize=(max(5, 3.2 * len(errors)), 4),
            constrained_layout=True,
        )
        for ax, metric in zip(axes.flat, errors):
            for (pathway, engine), group in audit.groupby(["pathway", "engine"]):
                ax.scatter(group["nodes"], group[metric].abs(), label=f"{pathway}/{engine}")
            ax.set(
                xlabel="Compute nodes",
                ylabel="Absolute magnitude",
                title=metric.removeprefix("metric_"),
            )
        axes.flat[0].legend(fontsize="small")
        fig.suptitle(f"{evidence_label} · numerical quality")
        fig.savefig(export_dir / "quality.svg")
        plt.show()
if not failed.empty:
    counts = failed.groupby(["pathway", "comparison"]).size().unstack(fill_value=0)
    ax = counts.plot.barh(stacked=True, figsize=(8, 3))
    ax.set(xlabel="Runs", ylabel="Pathway", title=f"{evidence_label} · excluded runs")
    ax.figure.tight_layout()
    ax.figure.savefig(export_dir / "excluded-runs.svg")
    plt.show()
if not pairs.empty:
    display(
        pairs[
            [
                "pathway",
                "method",
                "phase",
                "nodes",
                "reference_engine",
                "paired_allocations",
                "median_speedup",
                "iqr_speedup",
            ]
        ]
    )

## Phase time, scaling and resource use

Phase medians and IQR are across independent allocations. Weak-scaling efficiency accounts for observed workload growth. When the usual one-node baseline fails, each curve starts at its smallest successful allocation. Keep per-rank and per-worker memory scopes visible. Spill and task counts are available in the phase table when recorded.

In [ ]:
if not phase.empty:
    panel = phase.head(32).reset_index(drop=True)
    if len(phase) > len(panel):
        print(f"Showing {len(panel)} of {len(phase)} phase summaries. The CSV retains every row.")
    category = [
        f"{row.pathway}/{row.engine}/{row.method}/{row.phase}/{row.nodes}N/{row.scope}"
        for row in panel.itertuples()
    ]
    positions = np.arange(len(panel))
    fig, ax = plt.subplots(figsize=(10, max(3, 0.28 * len(panel))))
    ax.errorbar(
        panel["median_seconds"],
        positions,
        xerr=[
            panel["median_seconds"] - panel["q1_seconds"],
            panel["q3_seconds"] - panel["median_seconds"],
        ],
        fmt="o",
    )
    ax.set(
        yticks=positions,
        yticklabels=category,
        xlabel="Seconds",
        title=f"{evidence_label} · phase times",
    )
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(export_dir / "phase-timing.svg")
    plt.show()

if not scaling.empty:
    apply_points = scaling.loc[scaling["phase"] == "apply"]
    if not apply_points.empty:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
        for _, group in apply_points.groupby("series_id"):
            first = group.iloc[0]
            label = (
                f"{first.pathway}/{first.engine}/{first.method} "
                f"{first.scaling} (base {first.relative_to_nodes}N)"
            )
            scale_ax = axes[0 if first.scaling == "strong" else 1]
            scale_ax.plot(group["cores"], group["efficiency"], marker="o", label=label)
        for ax, title in zip(axes, ("Strong", "Weak")):
            ax.set(xlabel="Allocated cores", ylabel="Relative efficiency", title=title)
            ax.axhline(1, color="gray", linestyle="--")
            if ax.get_legend_handles_labels()[0]:
                ax.legend(fontsize="small")
        fig.suptitle(f"{evidence_label} · warmed apply")
        fig.savefig(export_dir / "scaling-efficiency.svg")
        plt.show()

if not phase.empty:
    memory = phase.dropna(subset=["peak_memory_bytes"])
    if not memory.empty:
        fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
        for (pathway, engine, scope), group in memory.groupby(
            ["pathway", "engine", "memory_scope"], dropna=False
        ):
            scope_label = scope if pd.notna(scope) else "scope unspecified"
            ax.scatter(
                group["cores"],
                group["peak_memory_bytes"] / 2**20,
                label=f"{pathway}/{engine} · {scope_label}",
            )
        ax.set(
            xlabel="Allocated cores",
            ylabel="Peak memory (MiB)",
            title=f"{evidence_label} · reported memory",
        )
        ax.legend(fontsize="small")
        fig.savefig(export_dir / "memory.svg")
        plt.show()

## Reuse and exports

Estimate cost as setup time, measured operator finalization where available, and the number of warmed applications multiplied by the warmed apply time. The estimate assumes the recorded warmed workload repeats and uses only matching phase scopes within a pair. Direct reuse measurements stay in the phase table. Blank memory or spill means unavailable. Charged core-hours may differ from the occupied-core estimate in the phase table.

In [ ]:
if not reuse.empty:
    display(
        reuse[
            [
                "pathway",
                "nodes",
                "comparison_pair_id",
                "reference_engine",
                "axis_setup_seconds",
                "axis_apply_seconds",
                "applications_to_break_even",
            ]
        ]
    )
    first = reuse.iloc[0]
    application_counts = np.array([1, 10, 100])
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.plot(
        application_counts,
        first.axis_setup_seconds + application_counts * first.axis_apply_seconds,
        marker="o",
        label="AXIS",
    )
    ax.plot(
        application_counts,
        first.reference_setup_seconds + application_counts * first.reference_apply_seconds,
        marker="o",
        label=first.reference_engine,
    )
    ax.set(
        xscale="log",
        xlabel="Applications",
        ylabel="Estimated seconds",
        title=f"{evidence_label} · reuse illustration ({first.comparison_pair_id})",
    )
    ax.legend()
    fig.savefig(export_dir / "reuse.svg")
    plt.show()

for name, table in {
    "audit": audit,
    "failed-or-unmatched": failed,
    "phases": phase,
    "paired-speedups": pairs,
    "scaling": scaling,
    "reuse": reuse,
}.items():
    table.insert(0, "evidence_label", evidence_label)
    table.to_csv(export_dir / f"{name}.csv", index=False)
print(f"Results exported to {export_dir}")

## Live worker validation

Use a validated Gaea or Ursa profile and worker-visible shared directory. From an approved scheduler host, run this command in the HELM checkout. It submits two Slurm worker jobs, verifies distinct compute hosts, and checks cleanup.

```bash
AXIS_RUN_SLURM=1 AXIS_SLURM_PROFILE=/path/to/site.json \
AXIS_SLURM_SHARED_DIR=/path/to/shared/scratch \
python -m pytest -q -s libs/axis/tests_python/test_axis_slurm_jobqueue.py
```